# Project 3 — Extending the U-Net (Outline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/project_3_outline.ipynb)

You finished Labs 06 and 07.  You have a working U-Net, a saved `unet_weights.pt`, and a
benchmark to compare against.  Now what?

This notebook is a **guide, not a lab**.  It does not run end-to-end.  It lists five
levels of extension you can pursue for your project, ordered roughly from gentle to
advanced.  Each level describes:

- **What to try** — the experiments in that direction.
- **Why** — what question it answers about your model.
- **Tricky parts** — the specific things that silently go wrong if you're not careful.
- **Skeleton** — minimal hints (often a few lines of pseudo-code) showing where to start.

Pick **one** level and go deep, or pick a couple of items from different levels.  Anything
you do should be measured on the **same train/test split** as Lab 06, with the **same
metrics** (mean Dice across classes 1 and 2, plus N/C ratio MAE).  Otherwise you cannot
honestly claim you improved over Lab 06.


## Prerequisites

Before you start any of the levels below, make sure you can:

1. Re-create the **exact** train/test split from Lab 06 (same `random_state=42`, same
   `has_nucleus` filter).  Every comparison in this guide assumes that split.
2. Load `unet_weights.pt` into a fresh `SmallUNet` and reproduce the test Dice from
   Lab 06's table.  If those numbers don't match, fix that first — otherwise you'll
   chase ghosts.
3. Save your model weights and your test Dice / N/C MAE for **every** experiment, with
   a short label.  By the end of your project you will have many runs and you will not
   remember which is which.

Keep a small results table in this notebook (or a CSV) as you go:

```
experiment              dice_mean   nc_mae   notes
lab06_baseline          ...         ...      reproduces unet_weights.pt
my_first_extension      ...         ...      ...
```


## Level 0 — Inspect What You Already Have

**No new training.**  Use the trained Lab 06 U-Net to understand its behaviour.  This is
the cheapest level and the one most students skip — which is a mistake, because most of
the ideas for Levels 1–4 come from looking carefully at what the current model gets wrong.

### What to try

- **Visualise the first-layer filters.**  The very first `Conv2d` learns 32 filters of
  shape `(3, 3, 3)` — three input channels (RGB), 3×3 kernel.  Plot each filter as a tiny
  RGB image.  What colours does the network look for first?
- **Inspect bottleneck feature maps.**  After the two pooling steps, the U-Net is working
  with `(128, 64, 64)` features.  Pick one image and visualise a handful of those 128
  channels as heatmaps.  Which channels seem to "fire" on nuclei?  On cytoplasm?
- **Failure-mode taxonomy.**  Sort the test set by Dice.  Look at the worst 10 images and
  *categorise* the errors: small-nucleus misses, bg/cyt confusion, leaking across cell
  boundaries, over-segmentation of artefacts, etc.  Count how many of each.
- **Per-class Dice histograms.**  Plot one histogram for class 1 (cytoplasm) and one for
  class 2 (nucleus) across the test set.  Is one class systematically worse?
- **Seed sensitivity.**  Re-train Lab 06 with three different `random_state` values for
  the optimiser/init (keep the data split fixed).  How much does test Dice swing?  This
  tells you the **noise floor** for any improvement you claim later.

### Tricky parts

- First-layer filters are a tensor with shape `(out_ch, in_ch, kH, kW) = (32, 3, 3, 3)`.
  To display each as an RGB image, transpose to `(out_ch, kH, kW, in_ch)` and **normalise
  each filter independently** to `[0, 1]` — otherwise the largest filter dominates and
  the rest look black.
- To grab intermediate feature maps, register a forward hook:
  `model.bottleneck.register_forward_hook(lambda m, inp, out: feats.append(out))`.
  Don't try to slice into `nn.Sequential`.
- "Sort by Dice" — be sure you're using the same per-image Dice as Lab 06's
  `unet_scores`, not a different averaging scheme.
- Seed sensitivity: change `torch.manual_seed(...)` *before* you instantiate the model,
  not after.  Initialisation happens in `__init__`.

### Skeleton

```python
# First-layer filters
W = model.enc1.block[0].weight.detach().cpu().numpy()   # (32, 3, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for k, ax in enumerate(axes.flat):
    f = W[k].transpose(1, 2, 0)
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f); ax.axis('off')
```

```python
# Forward hook for bottleneck features
feats = []
hook = model.bottleneck.register_forward_hook(
    lambda m, inp, out: feats.append(out.detach().cpu()))
_ = model(torch.from_numpy(X[test_idx[0]]).unsqueeze(0).to(device))
hook.remove()
# feats[0] has shape (1, 128, 64, 64)
```


## Level 1 — Tune Training

Same architecture as Lab 06.  Same data.  Vary **training** hyperparameters and the
**loss**, and measure the effect.

### What to try

- **Train longer.**  Lab 06 stops at 10 epochs.  Train to 30 or 50, but log **train Dice
  and test Dice every epoch**.  When does test Dice plateau?  When does it start falling
  while train Dice keeps rising?  That's overfitting — and it tells you how much
  regularisation (Level 2) you actually need.
- **Sweep learning rates.**  Try `1e-4`, `3e-4` (Lab 06 default), `1e-3`, `3e-3`.  Plot
  test Dice vs learning rate.
- **Loss weighting.**  Lab 06 uses `CE + Dice` with equal weight.  Try
  `α·CE + (1−α)·Dice` for `α ∈ {0.0, 0.25, 0.5, 0.75, 1.0}`.  Pure Dice often gives the
  best Dice score but uglier predictions; pure CE often does the opposite.
- **Class-weighted cross-entropy.**  Nucleus pixels are rare (~few percent of pixels).
  Their CE term is drowned by background.  Weight the CE loss by `1 / class_frequency`
  computed on the training set.

### Tricky parts

- **Always evaluate on the same split.**  Re-using `random_state=42` is mandatory.  If
  you change the split between runs you're measuring noise, not your hyperparameter.
- **More epochs without a test curve is meaningless.**  You can't claim "more epochs
  helps" if you only know train loss — at some point train loss keeps falling while
  test Dice falls too.  Plot both.
- **Class weights are computed from `y[train_idx]` only.**  Computing them on the full
  dataset leaks test-set information.  Compute pixel counts per class on the training
  masks, take `1 / count`, normalise to sum to `num_classes`, and pass into
  `nn.CrossEntropyLoss(weight=...)`.
- **Sanity-check class weights.**  If the weight for class 2 is 100× the weight for
  class 0, your loss is going to explode early.  Clip or rescale.

### Skeleton

```python
# Class-weighted cross-entropy
counts = np.array([(y[train_idx] == c).sum() for c in range(3)], dtype=np.float64)
weights = counts.sum() / (3 * counts)         # inverse-frequency, normalised to mean 1
weights = torch.tensor(weights, dtype=torch.float32, device=device)
ce_loss = nn.CrossEntropyLoss(weight=weights)
```

```python
# Per-epoch test Dice (call inside the training loop)
def eval_dice():
    model.eval()
    scores = []
    with torch.no_grad():
        for imgs, masks in test_loader:
            preds = model(imgs.to(device)).argmax(1).cpu().numpy()
            for p, m in zip(preds, masks.numpy()):
                scores.append((dice_score_np(p, m, 1) + dice_score_np(p, m, 2)) / 2)
    return float(np.mean(scores))
```


## Level 2 — Regularise and Vary the Architecture

Modest changes to the U-Net itself or to how it's trained.  Each item here is a small
deviation from Lab 06 — not a new architecture.

### What to try

- **Dropout in the decoder.**  Add `nn.Dropout2d(p=0.2)` between conv blocks.  Does it
  close the train/test Dice gap from your Level 1 curves?
- **Weight decay.**  Pass `weight_decay=1e-4` to Adam.  Compare the train/test gap.
- **Deeper U-Net.**  Add a third encoder/decoder level: channels go
  `32 → 64 → 128 → 256 (bottleneck)`.  Spatial sizes go `256 → 128 → 64 → 32`.  Mirror in
  the decoder.
- **Wider U-Net.**  Double every channel count (`64 → 128 → 256`).  Watch GPU memory.
- **Test-time augmentation (TTA).**  Predict on each test image *and* on its horizontal
  flip; flip the second prediction back; average the **probabilities**; argmax.  Free
  Dice improvement, no retraining.

### Tricky parts

- **Going deeper has a floor.**  `256/2/2/2 = 32`, fine.  `256/2/2/2/2 = 16`, getting
  marginal.  At a 16×16 bottleneck you've thrown away most of the spatial detail your
  decoder is supposed to recover from skip connections.  Don't go past 4 levels at
  256×256 input.
- **Decoder concat dimensions.**  If you add a level, the decoder block at level *k*
  takes `up_channels + skip_channels` as input.  Trace the channel counts on paper before
  you write the code, or PyTorch will throw shape errors at the `torch.cat`.
- **TTA averages probabilities, not argmax results.**  Averaging argmax (i.e. voting) is
  not an ensemble — and it doesn't give you a real probability distribution.  Apply
  `torch.softmax` first, average those, then argmax.
- **Dropout placement matters.**  Dropping out the encoder kills your skip connections'
  signal.  Stick to the decoder.

### Skeleton

```python
# Test-time augmentation
def predict_tta(model, img_t):
    model.eval()
    with torch.no_grad():
        p1 = torch.softmax(model(img_t), dim=1)
        p2 = torch.softmax(model(torch.flip(img_t, dims=[-1])), dim=1)
        p2 = torch.flip(p2, dims=[-1])     # flip the prediction back
    return ((p1 + p2) / 2).argmax(dim=1)
```


## Level 3 — Data Augmentation

You have **200 images**.  This is small.  Augmentation is the single highest-leverage
intervention available to you.

### What to try

- **Geometric augmentations** — applied to **both** the image and the mask:
  random horizontal flip, random vertical flip, random 90° rotations.
- **Photometric augmentations** — applied **only** to the image:
  brightness, contrast, hue jitter, slight Gaussian blur.
- Train your Lab 06 U-Net on the augmented stream for the same number of epochs.
  Measure test Dice and N/C MAE.
- Ablation: train with *only* geometric augs, then with *only* photometric augs, then
  with both.  Which one helps more?

### Tricky parts

- **The mask must receive the same geometric transform as the image.**  If you call
  `random_flip(img)` and `random_flip(mask)` independently, they get *different* random
  decisions and your image/mask pair is now misaligned.  Your training will silently
  corrupt.  Always pair the transforms inside `__getitem__`.
- **Use nearest-neighbour interpolation on the mask.**  Bilinear interpolation produces
  fractional class labels (e.g. `1.7`), which are nonsense for integer classes.  In
  PyTorch: `mode='nearest'`.  In Albumentations: `interpolation=cv2.INTER_NEAREST` for
  the mask.
- **Color jitter must never touch the mask.**  Apply photometric transforms to the image
  only, *after* the geometric transforms.
- **Augmentation belongs only on the training loader.**  Augmenting the test set makes
  your evaluation noisy and incomparable to Lab 06's number.
- **Don't augment so hard you destroy the signal.**  90° rotation is fine for cytology
  (no canonical orientation).  But blurring beyond `σ ≈ 1` will erase the nuclear edge
  you're trying to learn.

### Skeleton

```python
# Inside SegDataset.__getitem__
img  = torch.from_numpy(self.X[idx])             # (3, 256, 256)
mask = torch.from_numpy(self.y[idx].astype(np.int64))  # (256, 256)

if self.training:
    # Paired geometric transforms
    if torch.rand(1) < 0.5:
        img  = torch.flip(img,  dims=[-1])
        mask = torch.flip(mask, dims=[-1])
    if torch.rand(1) < 0.5:
        img  = torch.flip(img,  dims=[-2])
        mask = torch.flip(mask, dims=[-2])
    k = int(torch.randint(0, 4, (1,)))
    img  = torch.rot90(img,  k, dims=[-2, -1])
    mask = torch.rot90(mask, k, dims=[-2, -1])

    # Photometric — image only
    img = img * (0.9 + 0.2 * torch.rand(1))      # brightness scale
    img = img.clamp(0, 1)
```


## Level 4 — Transfer Learning with a Pretrained Encoder

Replace Lab 06's *from-scratch* encoder with one that has already been trained on
millions of natural images (ImageNet).  Use the
[`segmentation_models_pytorch`](https://github.com/qubvel-org/segmentation_models.pytorch)
library, which gives you a U-Net with a pretrained ResNet/EfficientNet/etc. encoder in
one line.

This is the most "industrial" of the five levels, and the closest to how segmentation is
done outside coursework.

### What to try

- A pretrained-encoder U-Net:
  `smp.Unet(encoder_name='resnet18', encoder_weights='imagenet', in_channels=3, classes=3)`.
- **Two-phase training.**  First freeze the encoder, train the decoder for a few epochs
  (the decoder is random and would drag the encoder around).  Then unfreeze and
  fine-tune the whole thing at a lower learning rate.
- Compare to Lab 06's from-scratch U-Net on the same train/test split.
- Try a few encoder choices: `resnet18`, `resnet34`, `efficientnet-b0`.  Bigger isn't
  always better with `n=200`.

### Tricky parts

- **ImageNet normalisation is mandatory.**  Pretrained encoders expect inputs normalised
  with `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`.  Your images are in
  `[0, 1]`.  If you skip this step, the pretraining is wasted — you've handed the
  encoder inputs that look nothing like what it was trained on.
- **Cytology is not ImageNet.**  Pretrained low-level features (edges, gradients, basic
  texture) still help, but the gain is smaller than on natural-image segmentation.  Don't
  expect a dramatic jump.  A modest improvement is the realistic outcome.
- **Encoder freezing.**  Freezing is a one-liner:
  `for p in model.encoder.parameters(): p.requires_grad = False`.  When you unfreeze,
  set them all back to `True` **and rebuild the optimiser**, otherwise the optimiser
  doesn't know about the now-trainable parameters.
- **Reduce the learning rate when you unfreeze.**  Fine-tuning at the same LR you used
  for the random decoder will destroy the pretrained features.  Drop by 10×.
- **The library has its own losses** in `smp.losses`, but your Lab 06 `combined_loss`
  works exactly the same way — keep it for consistency with your earlier results.

### Skeleton

```python
!pip install segmentation_models_pytorch --quiet
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=3,
).to(device)

# ImageNet normalisation — apply to images before they go into the model
IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IM_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
def normalise(imgs):  return (imgs - IM_MEAN) / IM_STD

# Phase 1 — freeze encoder
for p in model.encoder.parameters(): p.requires_grad = False
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
# ... train a few epochs ...

# Phase 2 — unfreeze, lower LR
for p in model.encoder.parameters(): p.requires_grad = True
opt = torch.optim.Adam(model.parameters(), lr=3e-5)
# ... train more epochs ...
```


## How to Pick a Path

If you only have time for one thing:

- **Want to understand the model?**  Start at Level 0.  Most project reports are weak
  because the student never *looked* at where the model fails — they just bolted on more
  layers.
- **Want the largest expected improvement per unit of effort?**  Level 3 (augmentation).
  The dataset is small and the architecture is already good.
- **Want to learn what production segmentation looks like?**  Level 4 (transfer learning).
- **Want to do the most thorough, defensible project?**  Combine: Level 0 to find your
  failure modes → Level 3 to address them → a focused write-up.

## How to Report Your Results

Every figure and number in your report must answer:

1. **What changed** vs Lab 06?  (one sentence)
2. **What stayed the same?**  (split, metrics, training length)
3. **By how much did Dice change?**  Mean and standard deviation across test images, not
   just a single mean.
4. **By how much did N/C MAE change?**  Dice can improve while N/C MAE gets worse —
   always report both.
5. **One qualitative example** — pick a test image where your change clearly helped, and
   one where it didn't.  Honesty about failures is worth more than another decimal point.

A two-page report with three honest comparisons beats a ten-page report with twenty
runs that disagree on the split.
